In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary
from torchmetrics import Accuracy
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.optim as optim

import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import optuna
from io import BytesIO
import csv
import time

c:\Users\Micro Lab ML\Desktop\Gediyon\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model retrival

In [2]:
# mlflow.login() # to connect to databricks servers
mlflow.set_tracking_uri("http://localhost:5000")  # connecting to local host

In [3]:
# Inference after loading the logged model
model_uri = "runs:/b99f8abefd074a788cb499a3b80fd58e/lowest_val_model"
loaded_model = mlflow.pytorch.load_model(model_uri)

In [4]:
# Access specific parameter or metric from a single run
run_id = "b99f8abefd074a788cb499a3b80fd58e"  # Replace with actual run ID
run = mlflow.get_run(run_id)

# Accessing a specific parameter and metric
print(f"Parameter -'learning_rate': {run.data.params['learning rate']}")
print(f"Parameter -'activation fun': {run.data.params['activation']}")
print(f"Parameter -'batch size': {run.data.params['batch_size']}")
print(f"Parameter -'nodes per layer': {run.data.params['hidden_units']}")
print(f"Parameter -'number of hidden layers': {run.data.params['n_layers']}")
print('\n')
print(f"Metric -'best val loss (RMSE)': {run.data.metrics['lowest_val_loss']}")
print(f"Metric -'test MAE': {run.data.metrics['test_MAE_norm']}")
print(f"Metric -'test_MAPE_norm': {run.data.metrics['test_MAPE_norm']}")

Parameter -'learning_rate': 0.0004306500882091994
Parameter -'activation fun': ReLU
Parameter -'batch size': 192
Parameter -'nodes per layer': 447
Parameter -'number of hidden layers': 6


Metric -'best val loss (RMSE)': 0.5219610628793944
Metric -'test MAE': 0.39076887982354747
Metric -'test_MAPE_norm': 0.07323510245346337


In [24]:
# Import the dataset

df = pd.read_csv('Processed_Randomized_data_Nov_01.csv', parse_dates = True, index_col=0)  #  Coil data

df = df.dropna()

# Sample data
X = df[['cmdCoilCurrent_1(A)','cmdCoilCurrent_2(A)','cmdCoilCurrent_3(A)','cmdCoilCurrent_4(A)',
        'cmdCoilCurrent_5(A)','cmdCoilCurrent_6(A)','cmdCoilCurrent_7(A)','cmdCoilCurrent_8(A)',
        'x', 'y', 'z']].values  # input current and position
y = df[['U','V','W']].values # label


# split the data into train and test

X_train_and_val, X_test, y_train_and_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42) # 15% for test

# Split data (90% train, 10% validation)
X_train, X_val, y_train, y_val = train_test_split(X_train_and_val, y_train_and_val, test_size=0.15, random_state=42) # 15% for validation

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)



# Create DataLoader for training and validation
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=100, shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=100, shuffle=False)

C:\Users\Micro Lab ML\AppData\Local\Temp\ipykernel_11464\3175324796.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('Processed_Randomized_data_Nov_01.csv', parse_dates = True, index_col=0)  #  Coil data


In [11]:
# Get cpu or gpu for training.
device = "cuda" if torch.cuda.is_available() else "cpu"

In [31]:
for input,target in test_loader:
    single_input = input.to(device)
    target = target.to(device)

    print(f"input: {single_input}")
    print(f"Target: {target}")
    break


# Measure prediction time
start_time = time.time()
predictions = loaded_model(single_input)
end_time = time.time()

inference_time = end_time - start_time
print(f"Prediction: {predictions}")
print(f"Inference time for 100 predictions: {inference_time:.6f} seconds")


input: tensor([[-6.3540, 10.1686,  8.6357,  ...,  0.1403, -0.1400,  0.1608],
        [-6.0520, 12.4120, -2.2270,  ..., -0.0518, -0.2068,  0.1731],
        [-4.2736, -3.4512,  3.1834,  ...,  0.0279, -0.0685,  0.1299],
        ...,
        [ 5.8413, -1.2948,  6.4918,  ..., -0.1073, -0.0968,  0.2466],
        [-9.4797,  4.5354,  0.3765,  ..., -0.0369, -0.0722,  0.1779],
        [-2.1434, -4.2640, -9.6856,  ...,  0.0241, -0.0606,  0.1861]],
       device='cuda:0')
Target: tensor([[ 1.4725e+00, -1.5654e+00,  1.4565e+01],
        [-6.0213e+00, -5.3415e-01,  3.7989e+00],
        [-6.0036e+00,  1.6165e+00, -5.8576e+00],
        [-5.9005e+00,  7.9333e+00, -9.8978e-02],
        [ 2.8110e-01, -2.3668e+00,  2.5028e+00],
        [ 1.3020e+00,  8.5938e+00, -3.9673e+00],
        [ 9.0450e-02,  7.3275e-01,  1.8600e+00],
        [ 4.4463e+00,  4.3560e-01,  5.6613e+00],
        [-2.8945e-01,  1.6740e-01,  1.1681e+01],
        [-1.7283e+01,  1.8263e+00, -1.0686e+01],
        [ 1.0166e+01,  1.2104e+01, -1

In [32]:
def metrics_fn(outputs, labels):

    MAE = torch.mean(torch.abs(labels-outputs),dim= 0, keepdim=True)
    labels = torch.abs(torch.where(labels == 0, torch.tensor(1e-6), labels)) # to avoid division by zero
    labels = torch.mean(torch.abs(labels), dim = 0, keepdim = True)
        
    MAEP = MAE/labels
    MAE_norm= torch.norm(MAE) # mean absolute error 
    MAEP_norm = MAE_norm/torch.norm(labels) # mean absolute error
    
    return MAE, MAE_norm, MAEP, MAEP_norm

In [33]:
MAE, MAE_norm, MAEP, MAEP_norm = metrics_fn(predictions, target)

print(f"MAE: {MAE}")
print(f"MAE_norm: {MAE_norm}")
print(f"MAEP: {MAEP}")
print(f"MAEP_norm: {MAEP_norm}")


MAE: tensor([[0.2195, 0.2875, 0.4082]], device='cuda:0', grad_fn=<MeanBackward1>)
MAE_norm: 0.5454084873199463
MAEP: tensor([[0.0512, 0.0652, 0.0571]], device='cuda:0', grad_fn=<DivBackward0>)
MAEP_norm: 0.05784511938691139
